In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
#pip install trino pandas openpyxl openai

from trino.dbapi import connect
from trino.auth import BasicAuthentication
import pandas as pd
import json
import os
from tqdm import tqdm


In [3]:


# các giá trị truyền vào
CATALOG="hive"

TRINO_SCHEMA= "bi_silver"
TRINO_TABLE_TYPE= 'BASE TABLE' 

# TRINO_SCHEMA= "bi_gold"
# TRINO_TABLE_TYPE= "VIEW"


LAYER = TRINO_SCHEMA.split("_")[1]

DOMAIN = ""
SEMANTIC_SCHEMA = ""

DEFAULT_SYNONYMS = ""
DEFAULT_COLUMN_TYPE = ""
DEFAULT_AGGREGATION = ""

ENUM_THRESHOLD = 50

OUTPUT_DIR = f"./tmp/metadata/{TRINO_SCHEMA}"


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR+"/columns", exist_ok=True)

In [4]:

# --- 1. CẤU HÌNH HỆ THỐNG ---
CONFIG = {
    'host': os.getenv("VCS_TRINO_HOST"),
    'port': int(os.getenv("VCS_TRINO_PORT")),
    'user': os.getenv("VCS_TRINO_USER"),
    'catalog': 'hive',
    'schema': TRINO_SCHEMA,  # Schema mục tiêu
    'password': os.getenv("VCS_TRINO_PASSWORD"),
}

trino_catalog = CONFIG['catalog']
trino_schema = CONFIG['schema']

def get_connection():
    return connect(
        host=CONFIG['host'],
        port=CONFIG['port'],
        user=CONFIG['user'],
        catalog=CONFIG['catalog'],
        http_scheme = os.getenv("VCS_TRINO_HTTP_SCHEME", "http"),
        auth=BasicAuthentication(CONFIG['user'], CONFIG['password']),
    )
    
def fetch_all(cursor, sql):
    cursor.execute(sql)
    return cursor.fetchall()
    

In [5]:

# ==============================
# CONNECT TRINO
# ==============================

conn = get_connection()

cursor = conn.cursor()

In [6]:

# ==============================
# GET TABLE LIST
# ==============================

sql_silver = f"""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = '{TRINO_SCHEMA}'
AND table_type = '{TRINO_TABLE_TYPE}'
"""

sql_gold_view = f"""
SELECT table_name
FROM information_schema.views
WHERE table_schema = '{TRINO_SCHEMA}'
"""
    
rows = fetch_all(cursor, sql_silver )

tables = [x[0] for x in rows]

print(f"Found {len(tables)} tables")

print("Total tables:", len(tables))


Found 37 tables
Total tables: 37


In [7]:

# ==============================
# TABLE METADATA
# ==============================

table_rows = []

for table in tables:
    if str(table).startswith("biz_"):
        continue
    
    schema_name = table.split("_")[0] or ""
    
    if table == "company_contacts":
        schema_name = "crm"
        
    if schema_name == "finance":
        schema_name = "biz"
    elif schema_name == "cx":
        schema_name = "cx_cso"
        
    table_rows.append({
        "layer": LAYER,
        "schema": schema_name ,
        "table": table,
        "source": f"{CATALOG}.{TRINO_SCHEMA}.{table}",
        "description": "",
        "keep": True
    })

df_tables = pd.DataFrame(table_rows)

table_file = os.path.join(OUTPUT_DIR, "tables.xlsx")
df_tables = df_tables.sort_values(by=["schema", "table"])
df_tables.to_excel(table_file, index=False)

print("Created:", table_file)


Created: ./tmp/metadata/bi_silver\tables.xlsx


In [8]:

# ==============================
# HELPER FUNCTIONS
# ==============================

def detect_column_type(data_type):

    numeric_types = [
        "integer","bigint","double",
        "decimal","float","real"
    ]

    if any(x in data_type for x in numeric_types):
        return "metric"
    else:
        return "dimension"


def detect_aggregation(column_type):

    if column_type == "metric":
        return "sum;avg;count"
    else:
        return "count;count_distinct"


def detect_pkey(column):

    if column == "id":
        return True

    if column.endswith("_id"):
        return True

    return False


def generate_synonyms(col):

    base = col.replace("_"," ")

    synonyms = [
        col,
        base
    ]

    return "; ".join(synonyms)


def detect_enum(table,col):

    try:

        sql = f"""
        SELECT COUNT(DISTINCT {col})
        FROM {TRINO_SCHEMA}.{table}
        """

        cursor.execute(sql)
        cnt = cursor.fetchone()[0]

        if cnt <= ENUM_THRESHOLD:

            sql = f"""
            SELECT DISTINCT {col}
            FROM {TRINO_SCHEMA}.{table}
            LIMIT {ENUM_THRESHOLD}
            """

            cursor.execute(sql)

            vals = [str(x[0]) for x in cursor.fetchall()]

            return " ; ".join(vals)

        else:
            return ""

    except:
        return ""



In [9]:
# ==============================
# PROCESS TABLES
# ==============================

for table in tqdm(tables):
    if str(table).startswith("biz_"):
        continue
    
    sql_silver = f"""
    SELECT
        column_name,
        data_type,
        CASE WHEN is_nullable='YES' THEN 1 ELSE 0 END as nullable
    FROM information_schema.columns
    WHERE table_schema='{TRINO_SCHEMA}'
    AND table_name='{table}'
    ORDER BY ordinal_position
    """

    cols = fetch_all(cursor, sql_silver )

    rows = []
    
    schema_name = table.split("_")[0] or ""
    
    if table == "company_contacts":
        schema_name = "crm"
        
    if schema_name == "finance":
        schema_name = "biz"
    elif schema_name == "cx":
        schema_name = "cx_cso"

    for column_name, data_type, nullable in cols:

        column_type = detect_column_type(data_type)

        aggregation = detect_aggregation(column_type)

        pkey = detect_pkey(column_name)

        synonyms = generate_synonyms(column_name)

        enum = detect_enum(table,column_name)

        row = {
            "layer": LAYER,
            "table": table,
            "col": column_name,
            "data_type": data_type,
            "nullable": nullable,
            "pkey": pkey,
            "fkey": "",
            "description": "",
            "keep": True,
            "enum": enum,
            "domain": DOMAIN,
            "synonyms": synonyms,
            "column_type": column_type,
            "aggregation": aggregation,
            "schema": schema_name
        }

        rows.append(row)
    
    if not rows:
        print(f"Not found columns for {table} - {sql_silver}")
        continue
    df = pd.DataFrame(rows)
    
    types_to_clear = {
        "boolean", "number", "bigint", "double", "int", "float",
        "timestamp(3)", "timestamp", "date", "timestamp(6)", "time", "decimal(28,2)", "integer"
    }
    col_names_to_clear = {
        "id", "address", "mobile", "email", "name", "note", "uuid", "presales_name", 
        "project_manager", "sale_admin", "contract_number",
        "customer_name", "agent_l1", "subject"
    }
    mask = (
        df["data_type"].str.lower().isin(types_to_clear)
        | df["data_type"].str.lower().str.startswith("decimal")
        | df["col"].str.lower().isin(col_names_to_clear)
        | df["col"].str.lower().str.endswith("_mobile")
        | df["col"].str.lower().str.endswith("_alias")
        | df["col"].str.lower().str.endswith("_email")
        | df["col"].str.lower().str.endswith("_id")
        | df["col"].str.lower().str.endswith("_code")
        | df["col"].str.lower().str.endswith("_address")
        | df["col"].str.lower().str.endswith("_at")
        | df["col"].str.lower().str.endswith("_index")
        | df["col"].str.lower().str.endswith("_date")
        | df["col"].str.lower().str.endswith("_username")
        | df["col"].str.lower().str.endswith("_note")
        | df["col"].str.lower().str.endswith("_uuid")
        
    )

    df.loc[mask, "enum"] = ""

    file_path = os.path.join(
        OUTPUT_DIR,
        f"columns/{table}.xlsx"
    )
    df = df.sort_values(by=["schema", "table", "col"])
    df.to_excel(file_path,index=False)

print("DONE")

100%|██████████| 37/37 [05:35<00:00,  9.07s/it]

DONE


## Gen docs

In [49]:
TABLE_USER_PROMPT="""
You are a senior data architect documenting a data warehouse.

Write a short and clear Vietnamese description for a table.

Requirements:
- Length: 1-3 sentences.
- Explain the business purpose of the table.
- Mention the main type of data stored.
- Mention the typical usage in analytics or reporting.
- Avoid technical details like column names.
- Use a formal tone suitable for a data catalog.

Table information:
Table name: {table}

Columns:
{column_list}

Output format:
Return ONLY the description text.
"""

In [71]:
COLUMN_USER_PROMPT="""
You are a senior data architect designing a semantic layer for a business data warehouse.

Your task is to generate metadata for a column.

Generate the following fields:

- description: A clear Vietnamese description of the column's business meaning.
- domain: The business domain the column belongs to (finance, crm, sales, product, hr, etc.)
- synonyms: Alternative names or phrases users may use when asking questions in Vietnamese or English.
- column_type: One of the following values:
    dimension
    metric
    time
    id
- aggregation: Suggested aggregation functions used in analytics.

Rules:

description
- Write 1 short sentence in Vietnamese.
- Explain the business meaning.
- Do NOT mention technical details.

domain
- Choose the most relevant business domain.

synonyms
- Provide 5-10 synonyms.
- Include Vietnamese and English variants.
- Use ";" as separator.
- Synonyms must include:
  + Vietnamese business terms
  + English business terms
  + Natural phrases users may use in BI questions

column_type
- dimension → category fields
- metric → numeric values used in calculation
- time → date/time columns
- id → identifiers or keys

aggregation
- metric → sum;avg;min;max
- dimension → count;count_distinct
- time → none
- id → count_distinct

Return result in the following JSON format:

{{
"description": "",
"domain": "",
"synonyms": "",
"column_type": "",
"aggregation": ""
}}

Table name: {table_name}

Column name: {column_name}

Data type: {data_type}

Table domain (if known): {table_domain}

Other columns in the table:
{column_list}
"""

In [51]:
def load_tables_metadata(file_path: str) -> pd.DataFrame:
    """
    Load tables metadata from Excel.
    """

    df = pd.read_excel(file_path, dtype=str)

    # df = df[df["keep"] == True]

    return df


def load_columns_metadata(folder_path: str) -> pd.DataFrame:
    """
    Load all column metadata from a folder of Excel files.
    """

    dfs = []

    for file in os.listdir(folder_path):

        if file.endswith(".xlsx"):

            path = os.path.join(folder_path, file)

            df = pd.read_excel(path, dtype=str)

            dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)

    # df_all = df_all[df_all["keep"] == True]

    return df_all


def build_metadata_model(tables_df, columns_df):
    """
    Join tables and columns metadata
    """

    df = columns_df.merge(
        tables_df,
        on=["layer", "table", "schema"],
        how="left",
        suffixes=("_col", "_table")
    )

    return df


def get_table_columns(columns_df, table_name: str):
    
    return columns_df[
        columns_df["table"] == table_name
    ]
    
    
def get_table_metrics(columns_df, table_name: str):
    
    df = columns_df[
        (columns_df["table"] == table_name) &
        (columns_df["column_type"] == "metric")
    ]

    return df

def get_table_dimensions(columns_df, table_name: str):
    
    df = columns_df[
        (columns_df["table"] == table_name) &
        (columns_df["column_type"] == "dimension")
    ]

    return df


def build_column_dictionary(columns_df):
    
    dictionary = {}

    for _, row in columns_df.iterrows():

        key = f"{row['table']}.{row['col']}"

        dictionary[key] = {
            "description": row["description"],
            "synonyms": str(row["synonyms"]).split(";"),
            "type": row["column_type"],
            "aggregation": row["aggregation"]
        }

    return dictionary


def build_semantic_schema(tables_df, columns_df):
    
    schema = {}

    for table in tables_df["table"]:

        cols = columns_df[columns_df["table"] == table]

        schema[table] = {
            "metrics": cols[cols["column_type"] == "metric"]["col"].tolist(),
            "dimensions": cols[cols["column_type"] == "dimension"]["col"].tolist(),
            "time": cols[cols["column_type"] == "time"]["col"].tolist(),
            "ids": cols[cols["column_type"] == "id"]["col"].tolist()
        }

    return schema


In [24]:
from BaseLLM import BaseLLM
from StepTimer import StepTimer

llm = BaseLLM()
timer = StepTimer()

In [ ]:
def generate_table_desc(row):
    llm_output = row.get("description")
    
    if llm_output is not None and len(llm_output) > 0:
        print("Has Data: "+ str(row["layer"]) +" - " + str(row["table"]))
        return row
    
    try:
        with timer.measure("generate_table_desc"):
            prompt = TABLE_USER_PROMPT.format(
                table=row["table"],
                column_list=row["column_list"]
            )
            print(str(row["layer"]) +" - " + str(row["table"]) )
            r = llm.chat(system_prompt="You are a senior data architect documenting a data warehouse.", user_prompt=prompt)
            row["description"] = r
    except Exception as e:
        print(str(e))
    return row


def generate_column_desc(row):
    llm_output = row.get("description")
    
    if llm_output is not None and len(llm_output) > 0:
        print("Has Data: "+ str(row["layer"]) +" - " + str(row["table"]))
        return row
    
    try:
        schema = row["schema"]
        if schema == "biz":
            schema = "finance"
        with timer.measure("generate_column_desc"):
            prompt = COLUMN_USER_PROMPT.format(
                table_name=row["table"],
                column_name=row["col"],
                data_type=row["data_type"],
                table_domain=row["schema"],
                column_list=row["column_list"],
            )
            print(str(row["layer"]) +" - " + str(row["table"]) + " - " + str(row["col"]))
            r = llm.chat_json(prompt)
            print(r)
            row["description"] = r.get("description", "")
            row["domain"] = r.get("domain", "")
            row["synonyms"] = r.get("synonyms", "")
            row["column_type"] = r.get("column_type", "")
            row["aggregation"] = r.get("aggregation", "")
    except Exception as e:
        print("Error: "  + str(e))
    return row


def add_column_list_to_tables(tables_df, columns_df):
    
    col_list = (
        columns_df
        .groupby("table")["col"]
        .apply(list)
        .reset_index()
        .rename(columns={"col": "column_list"})
    )

    tables_df = tables_df.merge(
        col_list,
        on="table",
        how="left"
    )

    return tables_df


def add_column_list_to_columns(columns_df):
    
    column_lists = (
        columns_df
        .groupby("table")["col"]
        .apply(list)
        .to_dict()
    )

    columns_df["column_list"] = columns_df["table"].map(column_lists)

    return columns_df

In [43]:
tables_df = load_tables_metadata("./tmp/metadata/bi_silver/tables.xlsx")

columns_df = load_columns_metadata("./tmp/metadata/bi_silver/columns/")

tables_df = tables_df.fillna("")
columns_df = columns_df.fillna("")

# metadata = build_metadata_model(tables_df, columns_df)

# semantic_schema = build_semantic_schema(tables_df, columns_df)

# print(semantic_schema["finance_cost_plan"])

In [44]:
print(len(columns_df) , len(tables_df))

714 31


In [45]:
tables_df = add_column_list_to_tables(tables_df, columns_df)
columns_df = add_column_list_to_columns(columns_df)

In [46]:
columns_df.head(2)

,layer,table,col,data_type,nullable,pkey,fkey,description,keep,enum,domain,synonyms,column_type,aggregation,schema,column_list
0,silver,company_contacts,company_key,varchar,1,False,,,True,,,company_key; company key,dimension,count;count_distinct,crm,"[company_key, company_name, created_at, displa..."
1,silver,company_contacts,company_name,varchar,1,False,,,True,,,company_name; company name,dimension,count;count_distinct,crm,"[company_key, company_name, created_at, displa..."


In [47]:
tables_df.head(1)

,layer,schema,table,source,description,keep,column_list
0,silver,biz,finance_actual_cost,hive.bi_silver.finance_actual_cost,,True,"[base_currency_amount, category_code, cost_gro..."


In [63]:
tables_df = tables_df.apply(generate_table_desc, axis=1)

silver - finance_actual_cost
[TIMER] generate_table_desc: 1.3833s
silver - finance_cash_collection
[TIMER] generate_table_desc: 1.5320s
silver - finance_cost_plan
[TIMER] generate_table_desc: 1.0532s
silver - finance_product_revenue_plan
[TIMER] generate_table_desc: 1.2719s
silver - finance_production_cost_allocations
[TIMER] generate_table_desc: 1.3360s
silver - finance_revenue_plan
[TIMER] generate_table_desc: 0.9892s
silver - company_contacts
[TIMER] generate_table_desc: 1.3787s
silver - crm_contract_allocations
[TIMER] generate_table_desc: 1.6444s
silver - crm_contracts
[TIMER] generate_table_desc: 1.3336s
silver - crm_deal_interested_products
[TIMER] generate_table_desc: 1.5215s
silver - crm_deal_quotation_products
[TIMER] generate_table_desc: 1.5678s
silver - crm_deal_quotations
[TIMER] generate_table_desc: 1.3912s
silver - crm_deal_reasons
[TIMER] generate_table_desc: 1.1325s
silver - crm_deals
[TIMER] generate_table_desc: 1.5431s
silver - crm_deals_allocation
[TIMER] generate_t

In [64]:
tables_df.to_excel("./tmp/metadata/bi_silver/out_tables.xlsx")

In [72]:
columns_df[:2].apply(generate_column_desc, axis=1)

Error: 'data_type'
Error: 'data_type'


,layer,table,col,data_type,nullable,pkey,fkey,description,keep,enum,domain,synonyms,column_type,aggregation,schema,column_list
0,silver,company_contacts,company_key,varchar,1,False,,,True,,,company_key; company key,dimension,count;count_distinct,crm,"[company_key, company_name, created_at, displa..."
1,silver,company_contacts,company_name,varchar,1,False,,,True,,,company_name; company name,dimension,count;count_distinct,crm,"[company_key, company_name, created_at, displa..."


In [ ]:
tables_df = load_tables_metadata( "./tmp/metadata/bi_gold/tables.xlsx")

columns_df = load_columns_metadata("./tmp/metadata/bi_gold/columns/")

metadata = build_metadata_model(tables_df, columns_df)

semantic_schema = build_semantic_schema(tables_df, columns_df)

print(semantic_schema["finance_cost_plan"])